# JEOL elemental maps (.pts)

Import required commands and define path were demo data is stored. Adapt this for your installation.

In [ ]:
from pathlib import PurePath
import matplotlib.plt as plt
from JEOL_eds import JEOL_pts
import JEOL_eds.utils as JU

base = PurePath("/home/alxneit/private/src/JEOL_eds")

Load example data (`data/128.pts`). This data set contains 50 frames with size 128x128 and 4000 energy channels. With the default data type (`uint16`) this results in about 6.5 GB! On top of this, there are also 50 images stored (one per frame) that will be used to compensate from sample movement. To minimize memory consumption, we first load only the meta data that contain the reference spectrum (sum over all frames and positions) and plot it to locate the maximum energy that contains relevant data.

In [ ]:
dc = JEOL_pts(base / "data" / "128.pts", only_metadata=True)
JU.plot_spectrum(dc.ref_spectrum, log_y=True)

We use an energy cutoff of 12keV (we miss the very weak line around 13.5 keV). This reduces the memory required by 2/3.

Load the data (individual frames) including the drift images. The data cube will contain 50 frames each 128x128 pixels and 1200 energy channels.

In [ ]:
dc = JEOL_pts(base / "data" / "128.pts", read_drift="yes", split_frames=True, E_cutoff=12.0)
dc.dcube.shape

Before we continue, check, if drift corrections are required. We have two options: First, make a mp4 movie with the drift image in the left pane and the total edx intensity in the right pane. Use the file name of the data but with `.pts` replaced by `mp4`. Check out the movie to realize that the sample jumps around a bit. In the output, only the last frame is shown.

In [ ]:
dc.make_movie()

Second, we calculate the image shift for each frame (from cross correlation with a reference) using the drift images (`align_src="drift_images"`) or data (`align_src="data"`) to which we might apply a Wiener filter (`filtered=True`). We use the 10th frame (`reference=9`) as reference. Ignore any warnings.

In [ ]:
dc.shifts(align_src="data", filtered=True, reference=9)

We could also visualize the shifts as 2D probability distribution (histogram).